In [24]:
import numpy as np
import torch
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Used device: {device}")

Used device: cuda


In [18]:
docs = [
    """RAG (Retrieval-Augmented Generation) łączy wyszukiwanie informacji (retrieval) z generacją tekstu. 
    W odróżnieniu od klasycznego podejścia, model nie działa w próżni - przed wygenerowaniem odpowiedzi wyszukuje relewantne dokumenty, 
    które służą jako kontekst wejściowy. Dzięki temu wiedza modelu nie jest statyczna i rzadziej dochodzi do halucynacji.""",
    
    """Ewaluacja klasyfikatorów opiera się na miarach takich jak Accuracy, Precision i Recall. 
    Accuracy to (TP + TN) / (TP + FP + TN + FN). Precision mówi: 'z tego co model uznał za pozytywne, ile faktycznie było pozytywne'. 
    F1-score to średnia harmoniczna precision i recall, stosowana gdy dane są niezbalansowane.""",
    
    """Metryka BLEU (Bilingual Evaluation Understudy) służy do oceny tłumaczeń i generacji. 
    Bazuje na precyzji n-gramów (od 1 do 4) i stosuje karę za zbyt krótkie zdania, czyli brevity penalty (BP). 
    Z kolei ROUGE skupia się na recallu - sprawdza, ile n-gramów referencji pokrywa wygenerowany tekst.""",
    
    """System BioZet-4 (fikcyjny) to technologia filtracji wprowadzona w 2025 r. przez AquaFuture. 
    Używa membran grafenowych, ale wymaga wymiany filtrów co 2 tygodnie i nie działa poniżej 5 stopni Celsjusza."""
]

queries = [
    "Czym różni się RAG od klasycznego podejścia w LLM?",
    "Jaki jest wzór na Accuracy?",
    "Czym różni się metryka BLEU od ROUGE?",
    "W jakich temperaturach nie działa BioZet-4?"
]

In [19]:
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=device)

def create_embeddings(texts: List[str]) -> np.ndarray:
    return embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)

def retrieve_top_k(query_emb: np.ndarray, doc_embs: np.ndarray, k: int = 2) -> List[int]:
    from sklearn.metrics.pairwise import cosine_similarity
    
    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)
        
    sims = cosine_similarity(query_emb, doc_embs)[0]
    top_indices = np.argsort(sims)[::-1][:k]
    return top_indices

doc_embeddings = create_embeddings(docs)
print("Documents embeddings created")

Documents embeddings created


In [20]:
model_name = "google/flan-t5-large"

print(f"Loading the model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)
print("Model loaded.")

Loading the model: google/flan-t5-large...
Model loaded.


In [21]:
def generate_text(prompt: str, max_new_tokens: int = 150) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,       
            do_sample=False,   
            repetition_penalty=1.2
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def run_experiment(query: str):
    prompt_no_rag = f"Pytanie: {query}\nOdpowiedź:"
    answer_no_rag = generate_text(prompt_no_rag)
    
    q_emb = create_embeddings([query])
    top_idx = retrieve_top_k(q_emb, doc_embeddings, k=2)
    retrieved_texts = [docs[i] for i in top_idx]
    
    context_str = "\n".join(retrieved_texts)
    prompt_rag = (
        f"Na podstawie poniższego kontekstu odpowiedz na pytanie.\n\n"
        f"Kontekst:\n{context_str}\n\n"
        f"Pytanie: {query}\n"
        f"Odpowiedź:"
    )
    answer_rag = generate_text(prompt_rag)
    
    return {
        "query": query,
        "retrieved": retrieved_texts,
        "answer_no_rag": answer_no_rag,
        "answer_rag": answer_rag
    }

In [23]:
import pandas as pd
from IPython.display import display, HTML

results = []

for q in queries:
    res = run_experiment(q)
    results.append(res)

for r in results:
    display(HTML(f"""
    <div style="border: 1px solid #ddd; padding: 20px; margin-bottom: 25px; border-radius: 8px; color: black; background-color: white; font-family: sans-serif;">
        <h3 style="color: black; margin-top: 0; border-bottom: 2px solid #333; padding-bottom: 10px;">Pytanie: {r['query']}</h3>
        
        <div style="background-color: white; padding: 15px; margin-bottom: 15px; border-left: 5px solid #ff4d4d; border-top: 1px solid #eee; border-right: 1px solid #eee; border-bottom: 1px solid #eee;">
            <strong style="color: black;">BEZ RAG (Wiedza ogólna):</strong><br>
            <div style="color: black; margin-top: 8px;">{r['answer_no_rag']}</div>
        </div>
        
        <div style="background-color: white; padding: 15px; border-left: 5px solid #2ecc71; border-top: 1px solid #eee; border-right: 1px solid #eee; border-bottom: 1px solid #eee;">
            <strong style="color: black;">Z RAG (Wiedza z dokumentów):</strong><br>
            <div style="color: black; margin-top: 8px;">{r['answer_rag']}</div>
        </div>
        
        <details style="margin-top: 15px;">
            <summary style="cursor: pointer; color: #555; font-weight: bold; font-size: 0.9em;">Zobacz źródła (Context)</summary>
            <div style="padding: 10px; color: #444; font-size: 0.85em; line-height: 1.5; background-color: #fafafa; border-radius: 4px; margin-top: 5px;">
                {'<hr style="border: 0; border-top: 1px solid #ddd; margin: 10px 0;">'.join(r['retrieved'])}
            </div>
        </details>
    </div>
    """))

## Komenatarz jakościowy
### 1. Eliminacja halucynacji i precyzja faktograficzna

* **Bez RAG:** Model wykazuje tendencję do generowania odpowiedzi ogólnikowych lub wręcz błędnych (np. przy pytaniu o BioZet-4 twierdzi jedynie, że nie działa w "bardzo niskich temperaturach").
* **Z RAG:** Model podaje precyzyjną wartość liczbową (**poniżej 5 stopni Celsjusza**), ponieważ ma dostęp do źródła. RAG zamienia "zgadywanie" na "odczytywanie".

### 2. Zgodność z dziedzinową bazą wiedzy (Wzory i definicje)

* **Bez RAG:** Przy pytaniach technicznych (Accuracy, BLEU/ROUGE) model bez kontekstu udziela odpowiedzi tautologicznych (np. "Accuracy odnosi się do dokładności pomiaru") lub całkowicie nielogicznych ("BLEU jest odpowiedzialne z ROUGE"), co czyni go bezużytecznym w zastosowaniach edukacyjnych czy specjalistycznych.
* **Z RAG:** Model poprawnie przytacza wzór matematyczny oraz wskazuje na kluczową różnicę między metrykami (Precision vs Recall).

### 3. Bariera językowa i kontekstualna

* **Bez RAG:** Oryginalny model (Flan-T5) mimo zapytań po polsku, często odpowiada po angielsku (np. "BioZet-4 is not working..."), co sugeruje, że jego "wiedza ogólna" w języku polskim jest uboższa.
* **Z RAG:** Dzięki dostarczeniu polskiego tekstu w kontekście, model generuje odpowiedź w języku polskim. Kontekst wymusza na modelu operowanie na konkretnym słownictwie, co drastycznie poprawia jakość interakcji w językach narodowych.

### 4. Aktualność wiedzy

* **Bez RAG:** Model nie wie nic o technologii z 2025 roku, ponieważ jego trening zakończył się wcześniej (knowledge cutoff).
* **Z RAG:** Model bez problemu operuje danymi o firmie AquaFuture i systemie BioZet-4, co pokazuje, że RAG pozwala na "aktualizację" wiedzy modelu bez konieczności jego ponownego trenowania.